## 1. Import the library

In [1]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
from sqlalchemy.exc import SQLAlchemyError
from pathlib import Path

In [2]:
# Load Data
base_dir = Path.cwd().parent
data_paths = {"olist_customer": base_dir / "data" / "raw" / "olist_customers_dataset.csv",
              "olist_geolocation": base_dir / "data" / "raw" / "olist_geolocation_dataset.csv",
              "olist_order_items": base_dir / "data" / "raw" / "olist_order_items_dataset.csv",
              "olist_order_payments": base_dir / "data" / "raw" / "olist_order_payments_dataset.csv",
              "olist_reviews" :base_dir / "data" / "raw" / "olist_order_reviews_dataset.csv",
              "olist_orders": base_dir / "data" / "raw" / "olist_orders_dataset.csv",
              "olist_products": base_dir / "data" / "raw" / "olist_products_dataset.csv",
              "olist_sellers": base_dir / "data" / "raw" / "olist_sellers_dataset.csv",
              "product_category": base_dir / "data" / "raw" / "product_category_name_translation.csv"}

In [3]:
# 1.1 Transform the data
def transform(df, table_name):
    original_len = len(df)
    
    # Chuẩn hóa tên cột: lowercase + bỏ khoảng trắng
    df.columns = df.columns.str.lower().str.strip().str.replace(' ', '_')
    
    # Xử lý datetime nếu có
    datetime_cols = [c for c in df.columns if 'date' in c or 'timestamp' in c]
    for col in datetime_cols:
        df[col] = pd.to_datetime(df[col], errors='coerce')
    
    return df

In [4]:
# 1.2 Connect database
def connect_database():
    username = 'root'
    password = '29092003'
    host = 'localhost'
    port = '3306'
    database = 'olist'

    try:
        engine = create_engine(f"mysql+pymysql://{username}:{password}@{host}:{port}/{database}")
        with engine.connect():
            print("Connected MySQL successfully!")
        return engine

    except SQLAlchemyError as e:
        print("Database connection failed!")
        print(e)
        return None

In [5]:
# 1.3 Loading data to database
def load_data(data_paths):
    engine = connect_database()

    if engine is None:
        return

    for table_name, path in data_paths.items():
        try:
            df_raw = extract(path)
            df_raw = transform(df_raw, table_name)
            df_raw.to_sql(name=table_name, con=engine, if_exists='replace', index=False)
            print(f"Loaded {table_name}")

        except Exception as e:
            print(f"Failed loading {table_name}")
            print(e)

In [6]:
load_data(data_paths)

Connected MySQL successfully!
Failed loading olist_customer
name 'extract' is not defined
Failed loading olist_geolocation
name 'extract' is not defined
Failed loading olist_order_items
name 'extract' is not defined
Failed loading olist_order_payments
name 'extract' is not defined
Failed loading olist_reviews
name 'extract' is not defined
Failed loading olist_orders
name 'extract' is not defined
Failed loading olist_products
name 'extract' is not defined
Failed loading olist_sellers
name 'extract' is not defined
Failed loading product_category
name 'extract' is not defined
